# Credit Card Fraud Detection in Azure Machine Learning

## Executive Summary

This notebook documents a proof-of-concept fraud detection workflow for a financial services audience. Its purpose is to explain, in plain language, how transaction data moves through an Azure Machine Learning pipeline to support the detection of potentially fraudulent credit card activity.

The model shown here is an **anomaly detection** model. Instead of relying only on known fraud labels, it learns what normal transaction behavior looks like and then flags unusual events for further review. This is useful in fraud operations because fraud patterns change over time and new attack methods may not match historical rules exactly.

For stakeholders, the main takeaway is simple: this workflow helps the organization move from raw transaction records to a prioritized set of suspicious transactions that analysts can investigate. It also makes clear that model performance must be judged in business terms, especially the trade-off between **false positives** (legitimate customers interrupted) and **missed fraud** (financial loss and customer harm).

## What This Notebook Covers

This notebook explains the full proof-of-concept process:

- connect to the Azure ML workspace and retrieve the registered dataset
- prepare the transaction data for modeling
- train an anomaly detection model
- evaluate the model with business-relevant metrics
- visualize the results for easier interpretation
- outline how the solution could be operationalized in Azure

## End-to-End Process at a Glance

```text
[Registered Transaction Dataset]
              ↓
 [Load into Notebook / Review Data]
              ↓
   [Prepare & Standardize Features]
              ↓
   [Train Anomaly Detection Model]
              ↓
 [Evaluate Fraud vs. Normal Outcomes]
              ↓
 [Visualize Results & Explain Drivers]
              ↓
 [Support Fraud Review and Deployment Planning]
```

## Azure ML Components and Their Roles

| Azure ML Component | Role in the Process |
|---|---|
| Azure ML Workspace | Central place to manage datasets, notebooks, experiments, and models |
| Registered Dataset | Stores the fraud transaction data used by the notebook |
| Notebook | Combines explanation, code, outputs, and visuals in one document |
| Compute Resource | Provides processing power when the notebook runs in Azure |
| Experiment Tracking | Helps compare runs and monitor model changes over time |
| Model Registry | Stores trained models for later reuse or deployment |
| Deployment Services | Can expose the model to downstream business systems when production-ready |

## How to Read This Notebook

This notebook is organized as a business-friendly walkthrough of the ML pipeline. Each section below explains **what the code is doing**, **why it matters**, and **what a non-technical stakeholder should take away**. The Python code itself is unchanged; the focus of this submission is on improving the explanation and decision-making context around the proof of concept.


## Step 1: Connect to the Azure Machine Learning Environment

The first code block loads the software packages required for the workflow and establishes a connection to Azure Machine Learning.

### Why this matters

Before any analysis can happen, the notebook needs access to:
- the Azure workspace where data and models are stored
- the Python tools used for preprocessing, modeling, and evaluation
- the visualization packages used later to explain results

### Stakeholder takeaway

This is the setup stage. It does not make any fraud decisions yet; it prepares the working environment so the organization can use a repeatable, governed process rather than ad hoc local files.


In [ ]:
# Step 1: Import Packages and Connect to your Azure Workspace
from azureml.core import Workspace, Dataset         # see https://pypi.org/project/azureml-core/
import pandas as pd                                 # see https://pandas.pydata.org/docs/
from sklearn.ensemble import IsolationForest        # see https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
from sklearn.metrics import classification_report   # see https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from azureml.core.model import Model                # see https://docs.microsoft.com/en-us/python/api/azureml-core/azureml.core.model?view=azure-ml-py 

## Step 2: Retrieve the Credit Card Fraud Dataset

This section pulls the registered transaction dataset from Azure ML into the notebook so it can be reviewed and analyzed.

### What is happening here

The code:
- connects to the Azure ML workspace
- locates the registered dataset by name
- converts it into a pandas dataframe for analysis
- displays a preview of the first rows

### Why this matters to the business

A fraud model is only as reliable as the data it is built on. Pulling the dataset from a governed Azure workspace helps ensure:
- the team is using the correct version of the data
- the process is repeatable across experiments
- analysts and technical teams are working from a shared source of truth

### Practical note

This dataset is relatively large. When run locally, loading the data may take several minutes because the file must be downloaded from Azure.


In [ ]:
# You only need to run this if you've imported this notebook to Azure AI Machine Learning Studio - Notebook,
# in which case you'll also need to upload the config.json file to the same directory as this notebook,
# and then execute this code to determine the current working directory.
import os
print("Current working directory:", os.getcwd())
print("Files in this directory:", os.listdir())


In [ ]:
# if you're running locally then use this ...
path = None

# alternatively, if you're running in Azure AI Machine Learning Studio - Notebook, then use this ...
# (make sure to upload the config.json file to the same directory as this notebook)
#  and then execute this code to determine the current working directory.
path='Users/[REPLACE-THIS-WITH-YOUR-USERNAME]/config.json'
ws = Workspace.from_config(path=path)
dataset = Dataset.get_by_name(ws, name='creditcard_fraud')
df = dataset.to_pandas_dataframe()
df.head()

## Step 3: Prepare the Data for Modeling

At this stage, the transaction data is reshaped into a form the anomaly detection model can use effectively.

### What is happening here

The code:
- standardizes the **Amount** field so it is on a scale similar to the other features
- separates the data into:
  - **X** = the transaction features used by the model
  - **y** = the actual known outcome, used only for evaluation
- removes fields that are less useful for this proof of concept

### Why this matters

Machine learning models perform better when important numeric features are on comparable scales. Without this step, a single field such as transaction amount could overpower other signals and reduce model quality.

### Stakeholder takeaway

This is the data preparation phase. It improves fairness and consistency in how transactions are assessed, which is especially important when a model will influence customer-impacting decisions.


In [ ]:
df['Amount'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
X = df.drop(columns=['Class', 'Time'])
y = df['Class']

## Step 4: Train the Anomaly Detection Model

The model used here is **Isolation Forest**, a common algorithm for anomaly detection.

### Business explanation

Instead of trying to memorize every type of fraud, Isolation Forest looks for transactions that behave differently from the majority of normal transactions. In fraud work, that is useful because new fraud tactics may not resemble known historical cases.

### Why this model was chosen for the proof of concept

- fraud is rare relative to normal activity
- anomalies can be easier to detect than fully classify
- the method is efficient enough for a large transaction dataset
- it provides a practical first step before trying more advanced supervised models

### Stakeholder takeaway

This model is best understood as a **screening tool**. It highlights unusual transactions that deserve attention, rather than acting as a final approval or denial engine on its own.


In [ ]:
model = IsolationForest(contamination=0.0017, random_state=42)
model.fit(X)
y_pred = model.predict(X)
y_pred = [1 if x == -1 else 0 for x in y_pred]

## Step 5: Evaluate Model Performance

After training, the model must be assessed against known outcomes to determine whether it is useful in practice.

### Key metrics in plain language

| Metric | What it tells stakeholders |
|---|---|
| Precision | Of the transactions flagged as fraud, how many were actually fraud? |
| Recall | Of the true fraud cases, how many did the model catch? |
| F1 Score | A balance between precision and recall |
| Support | How many cases were available in each class |

### Business interpretation of the reported results

The proof-of-concept results indicate that the model is very strong at recognizing normal transactions, but much weaker at identifying fraud. In practical terms, it catches only a minority of actual fraud cases.

This matters because a fraud operation usually cares about two opposing risks:

1. **False positives**  
   Legitimate customers may be interrupted, transactions may be declined, and analysts may waste time reviewing harmless cases.

2. **Missed fraud**  
   Fraudulent transactions pass through undetected, creating direct financial loss and customer trust issues.

### Stakeholder takeaway

The current model is a useful prototype, but it is **not yet production-ready as a stand-alone fraud control**. Its main value is to demonstrate the pipeline and reveal where future improvement is needed.


In [ ]:
# Step 5: Evaluate Model
print(classification_report(y, y_pred))

## Step 6: Optional Model Registration

This optional step stores the trained model in Azure ML so it can be managed more formally.

### Why this matters

Registering a model makes it easier to:
- version the model over time
- compare future improvements against this baseline
- prepare for controlled deployment later

### Stakeholder takeaway

This is a governance and lifecycle step. It does not improve model quality by itself, but it supports repeatability, auditability, and future deployment planning.


In [ ]:
import joblib                                       # see https://joblib.readthedocs.io/en/latest/
                                                    #     Joblib is a set of tools to provide lightweight pipelining in Python
joblib.dump(model, 'isolation_forest.pkl')
Model.register(model_path='isolation_forest.pkl',
               model_name='creditcard_if_model',
               workspace=ws)


## Step 7: Review the Number of Predicted Anomalies

The first chart summarizes how many transactions the model predicted as normal versus anomalous.

### Why this chart is useful

For executives and operations leaders, this acts as a quick reasonableness check:
- if the model flags far too many anomalies, analyst workload may become unmanageable
- if it flags almost none, important fraud cases may be missed
- if the flagged count is roughly aligned with expectations, the model may be behaving more realistically

### Stakeholder takeaway

This chart helps answer a practical question: **Is the model usable operationally, or will it overwhelm the fraud review team?**


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Add predictions to the original dataframe
df['predicted_anomaly'] = y_pred

# Count of predicted anomalies
sns.countplot(x='predicted_anomaly', data=df)
plt.title('Count of Predicted Anomalies')
plt.xlabel('Anomaly (1) vs Normal (0)')
plt.ylabel('Count')
plt.show()


## Step 8: Compare Transaction Amounts by Prediction Class

The box plot compares transaction amounts for transactions predicted as normal and those predicted as suspicious.

### Why this matters

Fraud models sometimes overreact to a single obvious feature, such as unusually large purchases. This chart helps test whether the model is relying too heavily on amount alone.

### Stakeholder takeaway

A good fraud model should not simply flag “big purchases.” It should consider multiple signals together. This visual helps the team decide whether additional feature engineering is needed.


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='predicted_anomaly', y='Amount')
plt.title('Transaction Amount by Prediction Class')
plt.show()


## Step 9: Explain Feature Influence with SHAP

The SHAP beeswarm plot helps explain which features most influenced the model's decisions.

### Why this matters for decision-makers

Fraud models are more credible when the team can explain, at least at a high level, **why** transactions were flagged. SHAP supports transparency by showing:
- which features mattered most
- whether high or low values pushed the model toward suspicious predictions
- whether the model appears to rely on sensible signals

### Stakeholder takeaway

This step supports responsible AI practice. Even if the model is not yet ready for production, this visual helps the team build trust, identify weak features, and plan future improvement work.


In [ ]:
import shap

explainer = shap.Explainer(model, X)
shap_values = explainer(X[:100])
shap.plots.beeswarm(shap_values)

## Business Impact Assessment and Next Steps

### Cost-Benefit Trade-off: False Positives vs. Missed Fraud

For a financial services company, neither error type is free:

- **False positives** create customer friction, call-center demand, analyst workload, and possible reputational damage.
- **Missed fraud** creates direct financial loss, chargebacks, investigation cost, and loss of trust.

In most real deployments, leadership would choose an operating threshold based on business priorities, customer experience tolerance, and fraud loss appetite.

### Recommendations for Improvement

This proof of concept should be improved before production use. Recommended next steps include:
- add richer business features such as merchant category, transaction velocity, geolocation, device fingerprint, and customer behavior history
- compare Isolation Forest against supervised models such as Random Forest, XGBoost, or ensemble approaches
- evaluate thresholds using business cost scenarios instead of accuracy alone
- test time-based validation so the model is judged on more realistic future-like data
- include human review for flagged cases during early deployment

### Risk Assessment and Mitigation

| Risk | Why it matters | Mitigation |
|---|---|---|
| Missed fraud | Direct financial loss and customer harm | Improve recall with better features and model comparison |
| Too many false alerts | Poor customer experience and wasted analyst time | Tune thresholds and use analyst feedback loops |
| Model drift | Fraud tactics change over time | Retrain and monitor regularly |
| Limited transparency | Harder to build trust in business settings | Use SHAP, clear dashboards, and documented review procedures |
| Overreliance on the model | Staff may assume alerts are always correct | Keep a human-in-the-loop review process |

### Stakeholder Communication Plan

When presenting this model to non-technical leaders, the message should be:
1. this is a **working proof of concept**
2. it shows the organization can operationalize an end-to-end fraud ML pipeline in Azure
3. current performance is not sufficient for stand-alone production use
4. the right next step is controlled improvement, business-focused testing, and staged rollout with analyst oversight

## Conclusion

This notebook demonstrates the complete fraud detection workflow in a form that non-technical stakeholders can understand. It shows not only how the model is built, but also how to judge whether it is trustworthy, useful, and aligned with business goals.
